# Lab: customer segmentation with RFM and k-means

The shop is a UK online retailer of gifts; many of its customers are wholesalers. One row of the raw table is one product line on one invoice. The marketing lead: "We send the same email to everyone. Give me a few groups of customers that behave differently, say how big each one is, and what you would send each."

Work through the checkpoints in order. C1 is the minimum, C2 is today's target, C3 is for those who finish C2.

## The starter

The cell loads the year from 2010-12-01 to 2011-12-09 with `Invoice` and `StockCode` read as text (an invoice number like `C536379` is not a number), and lists the stock codes that are not products. The renames make older downloads of the file use the same column names.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

raw = pd.read_csv("data/online_retail_ii.csv", dtype={"Invoice": str, "StockCode": str, "InvoiceNo": str})
raw = raw.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price", "CustomerID": "Customer ID"})
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
raw = raw[raw["InvoiceDate"] >= "2010-12-01"].reset_index(drop=True)

# Stock codes that are postage, fees, manual adjustments or tests, not products
NON_PRODUCT = {"POST", "D", "M", "C2", "DOT", "BANK CHARGES", "PADS", "CRUK", "AMAZONFEE", "S", "B", "ADJUST",
               "ADJUST2", "TEST001", "TEST002", "GIFT"}
print(len(raw), "rows from", raw["InvoiceDate"].min().date(), "to", raw["InvoiceDate"].max().date())
raw.head()

## C1 · One row per customer

### Step 1 · Clean the rows, one rule at a time

Count what each rule removes: rows with no `Customer ID`; cancellations (`Invoice` starts with `C`); rows with `Quantity` or `Price` not above 0; stock codes in `NON_PRODUCT`.

### Step 2 · Revenue and the snapshot date

Add `Revenue = Quantity * Price`. Set `snapshot` to the day after the last invoice.

### Step 3 · The RFM table

Group by `Customer ID`: **Recency** (days from the customer's last invoice to `snapshot`), **Frequency** (distinct invoices), **Monetary** (total revenue).

## C2 · Cluster and choose k

### Step 4 · Look, then log and scale

Look at the three columns with `describe()` and `skew()`. Then build `prep = make_pipeline(FunctionTransformer(np.log1p), StandardScaler())` and `Z = prep.fit_transform(rfm)`.

### Step 5 · Inertia and silhouette for k = 2 to 10

Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` on `Z` and record `inertia_` and `silhouette_score(Z, labels)`. Plot both against k.

### Step 6 · "How many groups?"

Where the inertia curve bends, what the silhouette score says, and how many groups the team can act on. Fit the k you chose.

## C3 · Name the segments

### Step 7 · Profile each segment

For your k: number of customers, median recency, median frequency, median monetary, and the segment's share of total revenue.

### Step 8 · Name each segment and give it an action

A table with four columns: name, size, what they look like, what to send.

## Bonus · k-means on the raw columns

## Check yourself